# 07 — Send Digest

Final step of the pipeline, runs after `load_to_mongo` succeeds.
Reads the same `articles` / `subscribers` collections the website reads,
picks the top-N trending articles, and emails them to every active
subscriber over plain SMTP (Gmail) — no third-party email service needed.

In [0]:
%pip install pymongo

In [0]:
%run "./00_config"

In [0]:
# 00_config should now have handed you MONGO_CONNECTION_STRING (whatever you
# already named it there) and GMAIL_APP_PASSWORD (add this one line to 00_config
# the same way you added the Mongo one — see the chat for how to get it).
# If your variable names differ, just point these two lines at the real ones.
mongo_connection_string = MONGO_URI
gmail_app_password = GMAIL_APP_PASSWORD

MONGO_DB = "fomoless"
FROM_EMAIL = "mouath1424@gmail.com"  # the Gmail address the app password belongs to
TOP_N = 8

In [0]:
import html
import smtplib
from datetime import datetime, timezone
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from pymongo import MongoClient

client = MongoClient(mongo_connection_string, tz_aware=True, serverSelectionTimeoutMS=8000, appname="fomoless-digest")
db = client[MONGO_DB]

In [0]:
# 1. Top trending articles. load_to_mongo just reported success, so an empty
#    result here means something's actually wrong — fail on purpose.
top_articles = list(
    db["articles"]
    .find({}, {"_id": 0, "Title": 1, "Url": 1, "Source_name": 1, "final_trend_score": 1})
    .sort([("final_trend_score", -1), ("Published_date", -1)])
    .limit(TOP_N)
)

if not top_articles:
    raise RuntimeError("No articles came back from 'articles'. Failing on purpose — this shouldn't be empty.")

In [0]:
# 2. Active subscribers. Zero is a normal state (nobody's signed up yet), not a bug.
subscribers = [d["email"] for d in db["subscribers"].find({"active": True}, {"_id": 0, "email": 1})]

if not subscribers:
    print("No active subscribers yet. Nothing to send today — exiting normally.")
    dbutils.notebook.exit("no_subscribers")

In [0]:
# 3. Compose the email once.
today = datetime.now(timezone.utc).strftime("%B %d, %Y")


def esc(text) -> str:
    return html.escape(str(text or ""))


rows = "".join(
    f"""
    <tr><td style="padding:14px 0;border-top:1px solid #E5E7E0;">
      <a href="{esc(a.get('Url', '#'))}" style="color:#17221B;font-size:16px;font-weight:600;text-decoration:none;">
        {esc(a.get('Title', 'Untitled'))}
      </a>
      <div style="color:#57655C;font-size:13px;margin-top:4px;">
        {esc(a.get('Source_name', ''))} &middot; trend score {float(a.get('final_trend_score') or 0):.2f}
      </div>
    </td></tr>"""
    for a in top_articles
)

html_body = f"""
<div style="font-family:Arial,sans-serif;max-width:560px;margin:auto;">
  <p style="font-family:monospace;letter-spacing:.1em;text-transform:uppercase;color:#4B7A1E;font-size:12px;">
    FoMoLess &middot; Daily Trend Radar</p>
  <h1 style="font-size:22px;margin:4px 0 18px;">Today on the radar — {today}</h1>
  <table style="width:100%;border-collapse:collapse;">{rows}</table>
  <p style="color:#8A968E;font-size:12px;margin-top:28px;">You subscribed at FoMoLess. Reply STOP to unsubscribe.</p>
</div>
"""
subject = f"FoMoLess — {len(top_articles)} trends for {today}"

In [0]:
# 4. Send over SMTP. One login, one connection reused for every subscriber —
#    not a new connection per email. Each subscriber gets their own message
#    (not one big BCC blast), which plays nicer with spam filters.
SMTP_HOST = "smtp.gmail.com"
SMTP_PORT = 587

with smtplib.SMTP(SMTP_HOST, SMTP_PORT, timeout=30) as server:
    server.starttls()
    server.login(FROM_EMAIL, gmail_app_password)
    for email_addr in subscribers:
        msg = MIMEMultipart("alternative")
        msg["Subject"] = subject
        msg["From"] = f"FoMoLess <{FROM_EMAIL}>"
        msg["To"] = email_addr
        msg.attach(MIMEText(html_body, "html"))
        server.sendmail(FROM_EMAIL, [email_addr], msg.as_string())

print(f"Sent today's digest ({len(top_articles)} articles) to {len(subscribers)} subscriber(s).")